In [1]:
# ---- 1. clone repo + install ----
!rm -rf /kaggle/working/Data-AI-Challenge   # clean any partial prior clone
!git clone https://github.com/nishanroy561/Data-AI-Challenge.git /kaggle/working/Data-AI-Challenge
!pip install -q sentence-transformers==3.0.1

Cloning into '/kaggle/working/Data-AI-Challenge'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 45 (delta 13), reused 39 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 80.99 KiB | 1.80 MiB/s, done.
Resolving deltas: 100% (13/13), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, bu

In [2]:
# ---- 2. wire imports ----
import sys, os, json, glob, gzip
import numpy as np
sys.path.append("/kaggle/working/Data-AI-Challenge")
!ls /kaggle/working/Data-AI-Challenge                  # sanity check — should list jd_config.py etc.
import jd_config as J
import features as F

CHECKLIST.md  notebooks      reasoning.py	     submission_metadata.yaml
docs	      precompute.py  requirements.txt	     validate_submission.py
features.py   rank.py	     sample_candidates.json
jd_config.py  README.md      space


In [3]:
# ---- 3. paths (auto-detect; works whether Kaggle keeps .gz or auto-extracts) ----
hits = sorted(glob.glob("/kaggle/input/**/candidates.jsonl*", recursive=True))
assert hits, "candidates file not found under /kaggle/input/ — attach the redrob-candidates dataset"
CANDIDATES = hits[0]
OUT = "/kaggle/working/artifacts"
os.makedirs(OUT, exist_ok=True)
print("Using:", CANDIDATES)

Using: /kaggle/input/datasets/nishanroy/redrob-candidates/candidates.jsonl


In [4]:
# ---- 4. build truthful narrative texts (summary + career, NOT skills) ----
ids, texts = [], []
opener = gzip.open if CANDIDATES.endswith(".gz") else open
with opener(CANDIDATES, "rt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            c = json.loads(line)
            ids.append(c["candidate_id"])
            texts.append(F.profile_text(c))
print(len(ids), "candidates")
print("sample:", texts[0][:200])

100000 candidates
sample: Backend Engineer | SQL, Spark, Cloud  Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spar


In [5]:
# ---- 5. embed on GPU ----
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = model.encode(texts, batch_size=512, show_progress_bar=True,
                   convert_to_numpy=True, normalize_embeddings=True)
jd = model.encode([J.JD_ANCHOR_TEXT], normalize_embeddings=True,
                  convert_to_numpy=True)[0]
print("embeddings:", emb.shape)

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-06-07 16:12:30.471433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780848750.702880      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780848750.765902      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780848751.306317      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

embeddings: (100000, 384)


In [6]:
# ---- 6. save artifacts ----
np.save(f"{OUT}/candidate_ids.npy", np.array(ids))
np.save(f"{OUT}/cand_embeddings.npy", emb.astype(np.float32))
np.save(f"{OUT}/jd_embedding.npy", jd.astype(np.float32))
with open(f"{OUT}/model.txt", "w") as fh:
    fh.write("BAAI/bge-small-en-v1.5\n")
print(f"Done. Artifacts in {OUT}. Save Version (top right) to commit them as a Kaggle output.")

Done. Artifacts in /kaggle/working/artifacts. Save Version (top right) to commit them as a Kaggle output.


In [7]:
!ls -lh /kaggle/working/artifacts/
import numpy as np
e = np.load("/kaggle/working/artifacts/cand_embeddings.npy")
jd_v = np.load("/kaggle/working/artifacts/jd_embedding.npy")
ids = np.load("/kaggle/working/artifacts/candidate_ids.npy")
print("emb:", e.shape, e.dtype, "| L2-normed:", round(float(np.linalg.norm(e[0])), 4))
print("jd: ", jd_v.shape, "| ids:", ids.shape, ids[0])

# Free smoke test: top-5 by raw cosine to JD
cos = e @ jd_v
import numpy as np
top = np.argsort(-cos)[:8]
for i in top:
    print(f"  {ids[i]}  cos={float(cos[i]):.3f}")

total 152M
-rw-r--r-- 1 root root 147M Jun  7 16:28 cand_embeddings.npy
-rw-r--r-- 1 root root 4.6M Jun  7 16:28 candidate_ids.npy
-rw-r--r-- 1 root root 1.7K Jun  7 16:28 jd_embedding.npy
-rw-r--r-- 1 root root   23 Jun  7 16:28 model.txt
emb: (100000, 384) float32 | L2-normed: 1.0
jd:  (384,) | ids: (100000,) CAND_0000001
  CAND_0055992  cos=0.896
  CAND_0018722  cos=0.889
  CAND_0088025  cos=0.884
  CAND_0061265  cos=0.884
  CAND_0041669  cos=0.884
  CAND_0041568  cos=0.884
  CAND_0086151  cos=0.883
  CAND_0014440  cos=0.883
